In [ ]:
! pip install numpy matplotlib gensim transformers torch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import gensim.downloader as api

import torch
from transformers import GPT2TokenizerFast, GPT2LMHeadModel

## 1. Recap

Weeks 1 and 2 built an agent that acts. Today's model does not act in a world at all. It
does exactly one thing: given some text, guess what comes next.

**The vocabulary.** A fixed set of $V$ tokens. GPT-3 uses $V = 50{,}257$.

**The input.** A sequence of token ids $(t_1, t_2, \ldots, t_n)$.

**The output.** A probability for every token in the vocabulary:

$$p(t_{n+1} \mid t_1, \ldots, t_n) = \text{softmax}(z), \qquad z \in \mathbb{R}^{V}$$

That is multi-class classification with an absurd number of classes. The scores $z$ are
called **logits**.

**Why the labels are free.** To train this we need pairs of (input, correct answer). But
the correct next token is already in the text: chop any document anywhere and the answer
is the token you chopped off. Nobody annotated anything, which is exactly why it is
possible to train on hundreds of billions of tokens.

Everything today builds toward one claim: **a transformer takes vectors that mean words
and repeatedly nudges them until they mean words in context.**

**What you will write.** Seven functions, spread across every section of this notebook:

| Section | Function | What it does |
|---|---|---|
| 2.1 | `cosine_similarity` | measures direction between two vectors |
| 2.2 | `nearest_neighbors` | finds the closest words in the vocabulary |
| 3.1 | `pca_project` | the projection from Q2 Week 7, rebuilt |
| 4.2 | `softmax` | turns scores into a distribution, stably |
| 4.3 | `attention_pattern` | the core of this session |
| 5.2 | `find_best_head` | searches GPT-2's 144 heads |
| 6.1 | `softmax_with_temperature` | the sampling dial |
| 6.4 | `generate` | the whole loop, end to end |

Each one is followed by a cell that checks it, and every later section calls the earlier
functions. Nothing here is optional and nothing is saved for the end: if you skip one, the
next section will not run.

### 1.1 Parameter Definitions

In [ ]:
RANDOM_SEED  = 178    # so everyone in the room gets the same numbers

GLOVE_MODEL  = "glove-wiki-gigaword-100"   # 400k words, 100 numbers each
VOCAB_LIMIT  = 30000  # search only the most common words, for speed
N_NEIGHBORS  = 6      # how many nearest words we print

D_K          = 16     # query and key dimension in our from-scratch attention
TOP_K        = 8      # how many next-token candidates we look at

SENTENCE     = ["a", "fluffy", "blue", "creature",
                "roamed", "the", "verdant", "forest"]

rng = np.random.default_rng(RANDOM_SEED)

### 1.2 The Data: Pretrained Word Vectors

A token id is just an index into a table, so it carries no meaning. Token 50,257 is not
"bigger" than token 1,832. The fix is to give every token a vector and let gradient
descent decide what goes in it.

GPT-3 uses 12,288 numbers per token. We use GloVe vectors with 100 numbers each, which
download quickly and behave much better for the similarity demos than raw transformer
token embeddings do.

The first run downloads about 130 MB and caches it.

In [ ]:
glove = api.load(GLOVE_MODEL)

print(f"vocabulary size: {len(glove.index_to_key)}")
print(f"vector length:   {glove.vector_size}")

### 1.3 What One Vector Looks Like

It is an ordinary array of floats. Nothing about it is human readable, and nobody chose
these numbers by hand.

In [ ]:
v = glove["king"]

print(f"type:  {type(v).__name__}")
print(f"shape: {v.shape}")
print(np.round(v[:12], 3))

## 2. Embedding Space

Two vectors that point the same way have a large dot product. Two that are perpendicular
have a dot product of zero. Dividing by the lengths removes the effect of magnitude, so
what is left is a pure measure of direction:

$$\text{cos}(a, b) = \frac{a \cdot b}{\|a\|\,\|b\|}$$

This is the same operation nearest centroid used in Q1, and the same one behind
$w^\top \Sigma w$ in the PCA session in Q2. It is also the only real mathematics inside
attention, which we get to in Section 4.

### 2.1 Cosine Similarity

Write the function. Some hints:

* `a @ b` or `np.dot(a, b)` gives you the numerator
* `np.linalg.norm(a)` gives you the length of a vector
* wrap the result in `float(...)` so it prints cleanly

In [ ]:
def cosine_similarity(a, b):
    """
    Cosine of the angle between two 1D vectors.

    Returns: a float in [-1, 1]
    """
    pass

#### Let's double check our functions!

In [ ]:
assert np.isclose(cosine_similarity([1, 0], [1, 0]), 1.0), \
    f"Expected 1.0. Actual: {cosine_similarity([1, 0], [1, 0])}"

assert np.isclose(cosine_similarity([1, 0], [0, 1]), 0.0), \
    f"Expected 0.0. Actual: {cosine_similarity([1, 0], [0, 1])}"

assert np.isclose(cosine_similarity([1, 0], [-1, 0]), -1.0), \
    f"Expected -1.0. Actual: {cosine_similarity([1, 0], [-1, 0])}"

assert np.isclose(cosine_similarity([1, 1], [1, 0]), 0.7071, atol=1e-4), \
    f"Expected 0.7071. Actual: {cosine_similarity([1, 1], [1, 0])}"

# magnitude must not matter, only direction
assert np.isclose(cosine_similarity([1, 2], [3, 4]), cosine_similarity([10, 20], [3, 4])), \
    "Scaling a vector should not change its cosine similarity with anything"

print("Passed")

### 2.2 Nearest Neighbours

If direction really does carry meaning, then the vectors nearest to a word should be words
you would call related. Precompute the search space first.

In [ ]:
VOCAB  = glove.index_to_key[:VOCAB_LIMIT]
MATRIX = np.array([glove[w] for w in VOCAB])

print(f"searching {len(VOCAB)} words, {MATRIX.shape[1]} numbers each")

Now write the search. Some hints:

* if `target` is a string, look it up with `glove[target]`, and add that word to the
  excluded set so a word is never its own nearest neighbour
* `zip(VOCAB, MATRIX)` pairs each word with its vector
* build a list of `(word, cosine_similarity(target, vec))` pairs, skipping any word that
  is excluded
* `scored.sort(key=lambda pair: pair[1], reverse=True)` puts the most similar first

In [ ]:
def nearest_neighbors(target, k=N_NEIGHBORS, exclude=()):
    """
    The k words in VOCAB pointing most nearly the same way as `target`.

    target:  a word in the vocabulary, or a vector
    exclude: words to leave out of the results

    Returns: a list of k (word, similarity) pairs, most similar first
    """
    pass

#### Let's double check our functions!

In [ ]:
hits = nearest_neighbors("king", k=3)

assert len(hits) == 3, f"Expected 3 results. Actual: {len(hits)}"

words_found = [w for w, _ in hits]
assert "king" not in words_found, \
    f"Expected 'king' to be excluded from its own neighbours. Actual: {words_found}"

sims = [sim for _, sim in hits]
assert sims == sorted(sims, reverse=True), \
    f"Expected the results sorted most similar first. Actual: {sims}"

assert "queen" not in [w for w, _ in nearest_neighbors("king", k=3, exclude=("queen",))], \
    "Expected 'queen' to be excluded when it is passed in `exclude`"

# a vector target excludes nothing, so the nearest word to paris is paris itself
word, sim = nearest_neighbors(glove["paris"], k=1)[0]
assert word == "paris", f"Expected 'paris'. Actual: {word}"
assert np.isclose(sim, 1.0), f"Expected 1.0. Actual: {sim}"

print("Passed")

Now use it.

In [ ]:
for word in ["king", "python", "irvine"]:
    print(f"\n{word}")
    for w, sim in nearest_neighbors(word):
        print(f"   {w:<15} {sim:.3f}")

### 2.3 Directions as Concepts

If the offset between *man* and *woman* really is a "gender direction", then adding it to
*king* should move us toward *queen*:

$$E(\text{king}) - E(\text{man}) + E(\text{woman}) \approx E(\text{queen})$$

In [ ]:
def analogy(a, b, c, k=5):
    """a is to b as c is to ___ . Computes b - a + c and finds its neighbours."""
    target = glove[b] - glove[a] + glove[c]
    return nearest_neighbors(target, k=k, exclude=(a, b, c))


for a, b, c in [("man", "king", "woman"),
                ("france", "paris", "japan"),
                ("walk", "walked", "swim")]:
    print(f"\n{a} : {b}  ::  {c} : ?")
    for w, s in analogy(a, b, c):
        print(f"   {w:<15} {s:.3f}")

Be skeptical of what you just saw.

Notice the `exclude` argument. Without it, the nearest vector to `king - man + woman` is
very often *king* itself, and the demo quietly drops the input words to avoid showing you
that. The analogies also work far better on GloVe than on the token embeddings inside a
transformer.

The claim that survives is the weaker one, and it is the only one attention needs: **the
space has learned structure, and that structure is directional.**

Try to break it. Some analogies fail badly.

In [ ]:
for a, b, c in [("small", "smaller", "large"),
                ("good", "better", "bad"),
                ("dog", "puppy", "cat"),
                ("water", "ice", "gold")]:
    top = analogy(a, b, c, k=3)
    answers = ", ".join(f"{w} ({s:.2f})" for w, s in top)
    print(f"{a} : {b} :: {c} : {answers}")

Some of those are right, some are near misses, and at least one is nonsense. The
vectors were trained on co-occurrence counts, not on grammar, so relationships that are
frequent in text survive and rarer ones do not.

## 3. Projecting Embeddings Into Two Dimensions

Last quarter we took four penguin measurements, projected them down to two dimensions
with PCA, and watched the three species separate without ever seeing a species label. The
structure was already in the geometry.

Here is the same function. The only thing that changes is what we hand it.

### 3.1 The PCA Function From Last Quarter

You wrote this in Q2 Week 7. Write it again, because it is worth seeing that nothing about
it changes when the rows stop being penguins and start being words. Some hints:

* centre the data by subtracting the column means, `X.mean(axis=0)`
* `np.cov(X_centered, rowvar=False)` gives the covariance matrix
* `np.linalg.eigh(cov)` returns eigenvalues in **ascending** order, so reverse the order
  with `np.argsort(eigvals)[::-1]`
* keep the first `n_components` eigenvectors as columns of `W`, then project with
  `X_centered @ W`
* return the projection and the eigenvalues, both in descending order

In [ ]:
def pca_project(X, n_components=2):
    """
    Project the rows of X onto its top principal components.

    Returns: Z of shape (n, n_components), and the eigenvalues in descending order
    """
    pass

#### Let's double check our functions!

In [ ]:
# all of the variance lies along the first axis, none along the second
X_test = np.array([[1.0, 0.0], [2.0, 0.0], [3.0, 0.0], [4.0, 0.0]])
Z_test, eig_test = pca_project(X_test, n_components=1)

assert Z_test.shape == (4, 1), f"Expected shape (4, 1). Actual: {Z_test.shape}"

assert np.isclose(eig_test[0], 5 / 3), f"Expected 1.6667. Actual: {eig_test[0]}"
assert np.isclose(eig_test[1], 0.0), f"Expected 0.0. Actual: {eig_test[1]}"

assert np.isclose(Z_test.mean(), 0.0), \
    f"Expected a centred projection with mean 0.0. Actual: {Z_test.mean()}"

# equal spacing in should give equal spacing out
assert np.allclose(np.abs(np.diff(Z_test[:, 0])), 1.0), \
    f"Expected gaps of 1.0 between projected points. Actual: {np.diff(Z_test[:, 0])}"

# eigenvalues must come back largest first
_, eig_desc = pca_project(np.random.default_rng(0).normal(size=(40, 5)))
assert np.all(np.diff(eig_desc) <= 1e-9), \
    f"Expected eigenvalues in descending order. Actual: {np.round(eig_desc, 3)}"

print("Passed")

### 3.2 Projecting a Curated Word Set

Pick words from a few clearly different neighbourhoods, project them, and see whether the
neighbourhoods survive the trip from 100 dimensions down to 2.

In [ ]:
groups = {
    "animals":   ["dog", "cat", "horse", "cow", "elephant", "tiger", "rabbit", "wolf"],
    "countries": ["france", "japan", "brazil", "canada", "egypt", "india", "norway", "kenya"],
    "food":      ["bread", "cheese", "rice", "apple", "chicken", "soup", "chocolate", "pasta"],
    "math":      ["algebra", "matrix", "theorem", "equation", "integral", "vector",
                  "probability", "geometry"],
}

words  = [w for group in groups.values() for w in group]
labels = [name for name, group in groups.items() for _ in group]

X = np.array([glove[w] for w in words])
Z, eigvals = pca_project(X, n_components=2)

print(f"{len(words)} words, {X.shape[1]} dimensions each")
print(f"variance captured by the first two components: {eigvals[:2].sum() / eigvals.sum():.1%}")

Colour by group, but remember that the projection never saw the groups. Any
separation is a property of the vectors alone.

In [ ]:
colors = {"animals": "crimson", "countries": "steelblue",
          "food": "seagreen", "math": "darkorange"}

plt.figure(figsize=(9, 6))
for name in groups:
    idx = [i for i, lab in enumerate(labels) if lab == name]
    plt.scatter(Z[idx, 0], Z[idx, 1], s=70, c=colors[name], label=name, zorder=3)

for i, w in enumerate(words):
    plt.annotate(w, (Z[i, 0], Z[i, 1]), fontsize=9, xytext=(4, 4), textcoords="offset points")

plt.xlabel("first principal component")
plt.ylabel("second principal component")
plt.title("GloVe vectors projected to 2D, coloured by a label PCA never saw")
plt.legend()
plt.tight_layout(); plt.show()

**Export this figure.** It is the one that goes on the "same projection, different
vectors" slide.

Two things worth noticing. The groups separate, but the axes do not mean anything you can
name, which is normal for PCA and is exactly what you saw with the penguins. And two
components capture only a small share of the total variance, so a lot of structure is
being flattened away. The separation you can see is the part that survived.

## 4. Attention From Scratch

A static embedding cannot be enough. The word *mole* gets one vector whether it means a
skin mark, an animal, or a unit of chemistry, because a lookup takes one token and returns
one vector with no reference to the rest of the sentence.

Attention fixes that. Every position produces a **query** (what am I looking for), every
position produces a **key** (what do I offer), we score every query against every key, and
the result decides how much each position contributes to each other position.

We use the sentence from lecture, and random projection matrices. Watch what that does.

### 4.1 Queries and Keys

$$q_i = W_Q\, e_i \qquad k_j = W_K\, e_j$$

Both are learned linear maps, the same object as a layer in the neural networks session.
Here they are random, because we have not trained anything.

In [ ]:
E = np.array([glove[w] for w in SENTENCE])   # (8, 100) static embeddings
d_model = E.shape[1]

W_Q = rng.normal(0, 1 / np.sqrt(d_model), size=(d_model, D_K))
W_K = rng.normal(0, 1 / np.sqrt(d_model), size=(d_model, D_K))

Q = E @ W_Q      # (8, 16) one query per position
K = E @ W_K      # (8, 16) one key per position

print(f"E: {E.shape}   Q: {Q.shape}   K: {K.shape}")

### 4.2 A Numerically Stable Softmax

We need a softmax that works row by row and survives `-inf` entries, since that is how
masking is implemented. Some hints:

* subtract the maximum along `axis` first, using `keepdims=True` so it broadcasts back.
  This changes nothing mathematically and stops `np.exp` from overflowing
* exponentiate, then divide by the sum along the same axis, again with `keepdims=True`
* `-inf` handles itself: after subtracting the max it stays `-inf`, and `np.exp(-inf)` is
  exactly 0

Everything after this point depends on this function, including Section 6.

In [ ]:
def softmax(x, axis=-1):
    """Softmax along one axis, stable against large values and -inf."""
    pass

#### Let's double check our functions!

In [ ]:
actual = np.round(softmax([2, 1, 0]), 4)
assert np.allclose(actual, [0.6652, 0.2447, 0.0900], atol=1e-4), \
    f"Expected [0.6652, 0.2447, 0.09]. Actual: {actual}"

# it must work row by row on a 2D array
rows = softmax([[1, 2], [3, 4]], axis=-1)
assert np.allclose(rows.sum(axis=1), 1.0), \
    f"Expected every row to sum to 1.0. Actual: {rows.sum(axis=1)}"

# masked entries must come out as exactly zero
masked = softmax([1.0, -np.inf])
assert np.allclose(masked, [1.0, 0.0]), f"Expected [1.0, 0.0]. Actual: {masked}"

# and large values must not overflow
big = softmax([1000.0, 999.0])
assert np.all(np.isfinite(big)), f"Expected finite probabilities. Actual: {big}"

print("Passed")

### 4.3 The Attention Pattern

Given queries `Q` and keys `K`, both of shape `(n, d_k)`, return the `(n, n)` attention
pattern. Row $i$ says where position $i$ looks:

$$\alpha_{ij} = \text{softmax}_j\!\left(\frac{q_i \cdot k_j}{\sqrt{d_k}}\right)$$

Three steps, and some hints:

* score every query against every key with `Q @ K.T`, then divide by `np.sqrt(d_k)` so the
  scores do not blow up in high dimensions
* if `causal`, set every score where $j > i$ to `-np.inf`, so a position cannot look at
  anything after itself. `np.tril` builds a lower triangular mask of ones and `np.where`
  applies it
* softmax along the last axis, so every row sums to 1

In [ ]:
def attention_pattern(Q, K, causal=True):
    """
    Row-stochastic attention weights.

    Q: (n, d_k) queries
    K: (n, d_k) keys

    Returns: (n, n) array whose rows each sum to 1
    """
    pass

#### Let's double check our functions!

In [ ]:
A_test = attention_pattern(Q, K)

assert A_test.shape == (len(SENTENCE), len(SENTENCE)), \
    f"Expected shape (8, 8). Actual: {A_test.shape}"

assert np.allclose(A_test.sum(axis=1), 1.0), \
    f"Expected every row to sum to 1.0. Actual: {A_test.sum(axis=1)}"

future = float(A_test[np.triu(np.ones_like(A_test, dtype=bool), k=1)].sum())
assert np.isclose(future, 0.0), f"Expected 0.0 mass on the future. Actual: {future}"

assert np.isclose(A_test[0, 0], 1.0), \
    f"Expected 1.0, the first position can only see itself. Actual: {A_test[0, 0]}"

# a case worked out by hand: Q = K = identity, d_k = 2
small = np.round(attention_pattern(np.eye(2), np.eye(2)), 4)
assert np.allclose(small, [[1.0, 0.0], [0.3302, 0.6698]], atol=1e-4), \
    f"Expected [[1.0, 0.0], [0.3302, 0.6698]]. Actual: {small}"

# turning the mask off should fill the whole grid
unmasked = float(attention_pattern(np.eye(4), np.eye(4), causal=False).sum())
assert np.isclose(unmasked, 4.0), f"Expected 4.0. Actual: {unmasked}"

print("Passed")

### 4.4 Drawing the Pattern

One helper, so we can draw our pattern and GPT-2's with the same code.

In [ ]:
def plot_attention(A, tokens, title, ax=None):
    """Heatmap of a causal attention pattern, with the weights written in."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7.5, 6))

    im = ax.imshow(np.where(A > 0, A, np.nan), cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha="right")
    ax.set_yticklabels(tokens)
    ax.set_xlabel("key: the word being looked at")
    ax.set_ylabel("query: the word doing the looking")
    ax.set_title(title)

    for i in range(len(tokens)):
        for j in range(i + 1):
            ax.text(j, i, f"{A[i, j]:.2f}", ha="center", va="center", fontsize=7.5,
                    color="white" if A[i, j] > 0.6 else "black")

    plt.colorbar(im, ax=ax, fraction=0.045)
    return ax

In [ ]:
A_random = attention_pattern(Q, K)

plot_attention(A_random, SENTENCE, "Attention with random W_Q and W_K")
plt.tight_layout(); plt.show()

Nothing meaningful happened. *Creature* does not attend to *fluffy*. *Forest* does not
attend to *verdant*. The rows are close to uniform over whatever the mask allows.

That is the lesson, and it is easy to miss: **the mechanism is not the meaning.** Every
step you implemented is exactly what GPT-2 does. What is missing is training. $W_Q$ and
$W_K$ only pick out adjective-to-noun relationships if the next-token loss made it
worthwhile for them to do so.

Change `RANDOM_SEED` and re-run if you like. You will not stumble into a linguistically
sensible head by luck.

## 5. A Real Attention Pattern

Now load GPT-2 and look at what a trained head does. GPT-2 small has 12 layers and 12
heads per layer, so there are 144 patterns to choose from for any input.

The first run downloads about 500 MB.

### 5.1 Loading GPT-2

We ask for the offset mapping as well as the token ids, because GPT-2 splits the less
common words into pieces and we will need to find a word by its position in the string
rather than by name.

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

try:
    model = GPT2LMHeadModel.from_pretrained("gpt2", attn_implementation="eager")
except TypeError:
    model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

text = " ".join(SENTENCE)
enc  = tokenizer(text, return_tensors="pt", return_offsets_mapping=True)
offsets = enc.pop("offset_mapping")[0].tolist()

with torch.no_grad():
    out = model(**enc, output_attentions=True)

tokens   = [tokenizer.decode([i]).strip() for i in enc["input_ids"][0]]
n_layers = len(out.attentions)
n_heads  = out.attentions[0].shape[1]

print(f"{n_layers} layers, {n_heads} heads each, {n_layers * n_heads} patterns")
print(f"{len(SENTENCE)} words became {len(tokens)} tokens: {tokens}")

### 5.2 Finding an Interpretable Head

Notice the token count. Eight words did not become eight tokens, which is Section 1.2 of
the lecture showing up in practice.

Most heads are not interpretable. Rather than guessing, search all 144 for the one where
*creature* pays the most attention to *fluffy* and *blue* combined. This is honest about
how head interpretation actually works: you go looking.

In [ ]:
def token_span(word):
    """Indices of the tokens covering `word`, since GPT-2 may split it into pieces."""
    start = text.index(word)
    end   = start + len(word)
    return [i for i, (a, b) in enumerate(offsets) if a < end and b > start]


noun       = token_span("creature")[-1]      # the last piece carries the context
adjectives = token_span("fluffy") + token_span("blue")

print(f"'creature' is token {noun}, the adjectives are tokens {adjectives}")

Now write the search itself. Some hints:

* `attentions[layer][0, head].numpy()` gives you the `(n, n)` grid for one head
* `A[query_pos, key_positions].sum()` adds up the weights that one row places on the
  columns you care about
* loop over every layer and every head, and keep the largest score you have seen
* return it together with the layer and head it came from

In [ ]:
def find_best_head(attentions, query_pos, key_positions):
    """
    Search every (layer, head) for the one where `query_pos` places the most
    attention on `key_positions`.

    attentions:    tuple of tensors, one per layer, each shaped (1, n_heads, n, n)
    query_pos:     the row of the grid we care about
    key_positions: the columns whose weights get added up

    Returns: (score, layer, head)
    """
    pass

#### Let's double check our functions!

In [ ]:
score, layer, head = find_best_head(out.attentions, noun, adjectives)

assert 0 <= layer < n_layers, f"Expected a layer in [0, {n_layers}). Actual: {layer}"
assert 0 <= head < n_heads, f"Expected a head in [0, {n_heads}). Actual: {head}"

reported = out.attentions[layer][0, head].numpy()[noun, adjectives].sum()
assert np.isclose(score, reported), \
    f"Expected the score to match that head's grid, {reported:.4f}. Actual: {score:.4f}"

every_score = [out.attentions[L][0, H].numpy()[noun, adjectives].sum()
               for L in range(n_layers) for H in range(n_heads)]
assert np.isclose(score, max(every_score)), \
    f"Expected the largest of all {len(every_score)} heads, {max(every_score):.4f}. Actual: {score:.4f}"

print("Passed")

In [ ]:
print(f"best head: layer {layer}, head {head}   adjective mass on 'creature': {score:.2f}")

In [ ]:
A_real = out.attentions[layer][0, head].numpy()

plot_attention(A_real, tokens, f"GPT-2, layer {layer}, head {head}")
plt.tight_layout(); plt.show()

### 5.3 Browsing Other Heads

Compare that with your random-weight plot. Same operation, same shapes, wildly different
structure.

Some heads attend almost entirely to the first token, a well documented behaviour
sometimes described as the head switching itself off. Others attend to the previous token,
which is close to what a small n-gram model would do.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))

for ax, (L, H) in zip(axes, [(0, 0), (1, 4), (5, 7)]):
    plot_attention(out.attentions[L][0, H].numpy(), tokens, f"layer {L}, head {H}", ax=ax)

plt.tight_layout(); plt.show()

None of these were designed. Nobody wrote a rule saying "attend to adjectives". These
patterns are what minimizing next-token loss produced, and reading them after the fact is
an active research area rather than a solved one.

## 6. Sampling and Temperature

After the last block, the vector at the final position is multiplied by the unembedding
matrix to give one logit per vocabulary token. Softmax turns those into probabilities.

Temperature divides the logits before the softmax:

$$p = \text{softmax}(z / T)$$

Small $T$ sharpens the distribution toward the argmax. Large $T$ flattens it toward
uniform. This is the exploration and exploitation tradeoff from Weeks 1 and 2, applied to
token selection instead of action selection.

### 6.1 Softmax With Temperature

You already wrote `softmax` in Section 4.2. This is one line on top of it.

In [ ]:
def softmax_with_temperature(logits, T=1.0):
    """
    Turn logits into a probability distribution, scaled by temperature T.

    Returns: an array the same shape as logits, summing to 1
    """
    pass

#### Let's double check our functions!

In [ ]:
# T = 1 is the plain softmax
actual = np.round(softmax_with_temperature([2, 1, 0], T=1.0), 4)
assert np.allclose(actual, [0.6652, 0.2447, 0.0900], atol=1e-4), \
    f"Expected [0.6652, 0.2447, 0.09]. Actual: {actual}"

# low temperature sharpens toward the argmax
actual = np.round(softmax_with_temperature([2, 1, 0], T=0.5), 4)
assert np.allclose(actual, [0.8668, 0.1173, 0.0159], atol=1e-4), \
    f"Expected [0.8668, 0.1173, 0.0159]. Actual: {actual}"

# high temperature flattens toward uniform
actual = np.round(softmax_with_temperature([2, 1, 0], T=2.0), 4)
assert np.allclose(actual, [0.5065, 0.3072, 0.1863], atol=1e-4), \
    f"Expected [0.5065, 0.3072, 0.1863]. Actual: {actual}"

# it is still a probability distribution at any temperature
actual = float(softmax_with_temperature([5, -3, 0.2, 1.7], T=0.3).sum())
assert np.isclose(actual, 1.0), f"Expected 1.0. Actual: {actual}"

print("Passed")

### 6.2 A Real Next-Token Distribution

Feed GPT-2 the prompt from the lecture slide and look at the top candidates.

In [ ]:
prompt = "This movie was"
ids = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    logits = model(**ids).logits[0, -1].numpy().astype(float)   # all 50,257 tokens

top   = np.argsort(logits)[::-1][:TOP_K]
probs = softmax_with_temperature(logits, T=1.0)

print(f'"{prompt} ___"   ({TOP_K} of {len(logits)} candidates)\n')
for i in top:
    print(f'   "{tokenizer.decode([i])}"{"":<8} {probs[i]:.1%}')

### 6.3 Turning the Temperature Dial

Same logits, three temperatures. Watch the same eight candidates redistribute.

In [ ]:
labels = [tokenizer.decode([i]).strip() for i in top]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, T in zip(axes, [0.2, 1.0, 1.8]):
    p = softmax_with_temperature(logits, T=T)[top]
    p = p / p.sum()                       # renormalise over the eight we are showing

    ax.bar(range(TOP_K), p, color=["crimson"] + ["steelblue"] * (TOP_K - 1))
    ax.set_xticks(range(TOP_K))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_title(f"T = {T}")
    ax.set_ylim(0, 1)

axes[0].set_ylabel("share of the top-8 mass")
plt.tight_layout(); plt.show()

### 6.4 The Generation Loop

Sample a token, append it, run again. There is no separate generation algorithm: an essay
is this loop run a few thousand times, with the model re-reading everything it has already
written on every single step.

Write the loop. Some hints:

* start from `ids = tokenizer(prompt, return_tensors="pt")["input_ids"]`
* inside `with torch.no_grad():`, `model(ids).logits[0, -1]` is the logit vector for the
  next token. Bring it across with `.numpy().astype(float)`
* turn it into probabilities with your `softmax_with_temperature`, then renormalise with
  `p = p / p.sum()` so numpy does not complain about float rounding
* `gen.choice(len(p), p=p)` draws one token id from that distribution
* append it with `torch.cat([ids, torch.tensor([[next_id]])], dim=1)` and go round again
* `tokenizer.decode(ids[0])` at the end turns the whole thing back into text

In [ ]:
def generate(prompt, n_tokens=25, T=0.8, seed=RANDOM_SEED):
    """Sample n_tokens continuations of the prompt, one token at a time."""
    pass

#### Let's double check our functions!

In [ ]:
seed_text = "The capital of France is"
sample = generate(seed_text, n_tokens=5, T=0.2)

assert isinstance(sample, str), f"Expected a string. Actual: {type(sample).__name__}"

assert sample.startswith(seed_text), \
    f"Expected the output to begin with the prompt. Actual: {sample!r}"

n_before = len(tokenizer(seed_text)["input_ids"])
n_after  = len(tokenizer(sample)["input_ids"])
assert n_after == n_before + 5, f"Expected {n_before + 5} tokens. Actual: {n_after}"

# same seed, same text
assert generate(seed_text, n_tokens=5, T=0.2) == sample, \
    "Expected the same seed to give the same continuation"

print("Passed")

In [ ]:
for T in [0.2, 0.8, 1.5]:
    print(f"--- T = {T} ---")
    print(generate("The best thing about machine learning is", T=T), "\n")

Low temperature is repetitive and often loops. High temperature wanders off topic and
eventually stops making sense. The useful range sits in between, and that is a choice you
make rather than something the model decides.

One more thing this makes concrete: nothing in the loss function rewards being *true*. The
objective rewards plausible continuations, and a confident false sentence looks the same as
a confident true one unless the training data reliably distinguished them.

## 7. Recap

1. **Cosine similarity** measures direction, and it is the operation behind nearest
   centroid, PCA, and attention alike. You built the neighbour search on top of it.
2. **Word vectors have learned structure.** Projecting them with last quarter's PCA
   function separates semantic groups that the projection never saw labels for.
3. **Attention is a small amount of code.** Queries, keys, a scaled dot product, a causal
   mask, and a softmax.
4. **The mechanism is not the meaning.** With random weights the same code produces
   nothing; the interpretable patterns in GPT-2 came entirely from training.
5. **Temperature is exploration versus exploitation**, and generation is one loop around a
   single next-token distribution.

If you want to keep going:

* Run the head search on a sentence of your own. Do the same heads light up?
* Section 4 skipped the value matrix. Add `W_V`, compute `A @ (E @ W_V)`, and add it back
  onto `E`. That is one full attention block.
* Compare the top-8 next tokens for "The capital of France is" against "The capital of
  Australia is". GPT-2 gets one of them wrong.